### Post processing of deviation scores from Normative models

These processing steps take inspiration from Rutherford et al. (2023).

This notebook consists of two tests:

1. Plotting of extreme deviations (-2>z>2)
    - Violin plot
2. Regression with support vector classification (SVC)
    - Implemented via scikit-learn

**Text is outdated**

**Reading and organising data**

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# Setting data directory variables
nmodel_results_dir = Path("nmodel_results")
# nmodel_results_dir.mkdir(exist_ok=True)

data_version_dir = nmodel_results_dir / "agg_data"
# data_version_dir.mkdir(exist_ok=True)
if not data_version_dir.exists():
    print(f"Path {data_version_dir} does not exist")
print(data_version_dir)

In [ ]:
# Setting directory to save results
postproc_res_dir = Path("postproc_results")
postproc_res_dir.mkdir(exist_ok=True)

postproc_res_dir_ver = postproc_res_dir / "agg_data" # Data specific path
postproc_res_dir_ver.mkdir(exist_ok=True)


In [ ]:
## Read data

# Clinical groups and ROIs of interest
groups = ["HC_train", "HC_test", "MDD"]
rois = ['brain_stem', 'left_amygdala', 'left_caudate',
       'left_thalamus', 'right_amygdala', 'right_caudate', 'right_thalamus',
       'ctx_lh_rostralanteriorcingulate', 'left_putamen', 'right_putamen',
       'ctx_rh_rostralanteriorcingulate', 'ctx_rh_rostralmiddlefrontal',
       'ctx_lh_rostralmiddlefrontal']

data_dict = {group : {} for group in groups}
print(data_dict)


for roi in rois:
    for group in groups:
        dir = data_version_dir / f"model_results_{roi}" / f"data/{group}.csv"
        df = pd.read_csv(dir, header=[0,1])
        df.insert(1, "subject_ids", df.pop("subject_ids"))
        data_dict[group][roi] = df






In [ ]:
## Make dataframes with Z-scores acros all ROI for each group.

groups = ["HC_train", "HC_test", "MDD"]

data_z_dict = {}

for group in groups:
    merged = None
    subject_sets = {}

    for roi in rois:
        df = data_dict[group][roi]

        # subject ids
        subject_id = df[("subject_ids", "")].rename("subject_id")

        # Z-table for this bin
        z = df["Z"].copy()

        # attach subject id and use it as row key
        z = pd.concat([subject_id, z], axis=1)
        z = z.drop_duplicates(subset="subject_id")
        z = z.set_index("subject_id")

        # optional check: record subject set per bin
        subject_sets[roi] = set(z.index)

        merged = z if merged is None else merged.join(z, how="outer")


    data_z_dict[group] = merged.reset_index()


In [ ]:
print(data_z_dict["HC_train"].shape)
print(data_z_dict["HC_test"].shape)
print(data_z_dict["MDD"].shape)

In [ ]:
data_z = pd.concat(
    [
        data_z_dict["HC_train"].assign(group="HC_train"),
        data_z_dict["HC_test"].assign(group="HC_test"),
        data_z_dict["MDD"].assign(group="MDD"),
    ],
    axis=0,
    ignore_index=True,
)

In [ ]:
print(data_z.shape)
print(data_z[data_z["group"] == "HC_train"].shape)
print(data_z[data_z["group"] == "HC_test"].shape)
print(data_z[data_z["group"] == "MDD"].shape)

## Support Vector Machine

In [ ]:
print(data_z[data_z["group"] == "HC_test"].isna().sum())
print(data_z[data_z["group"] == "MDD"].isna().sum())
print("Number of MDD and HC_test sub:", len(data_z[data_z["group"] != "HC_train"]))
print("Number of HC_test sub:", len(data_z[data_z["group"] == "HC_test"]))
print("Number of MDD sub:", len(data_z[data_z["group"] == "MDD"]))

**Excluding high Na ROIs**

In [ ]:
# Excludes high Na ROIs. Keeps more subjects
## Subsetting dataset for SVM input
rois_to_drop = ["ctx_lh_rostralmiddlefrontal", "ctx_rh_rostralmiddlefrontal"]

intermediate_df = data_z.copy()
intermediate_df = intermediate_df[intermediate_df["group"] != "HC_train"]
print(intermediate_df.shape)
intermediate_df = intermediate_df.drop(columns = ["subject_id", *rois_to_drop]).dropna()
print(intermediate_df.shape)

X = intermediate_df.drop(columns=["group"]).to_numpy(dtype=float)
Y = intermediate_df["group"].to_numpy(dtype=str)

print(X.shape)
print(Y.shape)


**Including high Na ROIs**

In [ ]:
# # Block for not excluding hig na ROIs. Keeps less subjects
# data_z2 = data_z.copy()
# data_z2 = data_z2.dropna()

# X_df = data_z2.copy()
# X_df = X_df[X_df["group"] != "HC_train"]
# print(X_df.shape)
# X = X_df.drop(columns=["subject_id", "group"]).to_numpy(dtype=float)
# print(X.shape)

# Y_df = data_z2.copy()
# Y_df = Y_df[Y_df["group"] != "HC_train"]
# Y = Y_df["group"].to_numpy(dtype=str)

In [ ]:
roi_mapping = {
    "brain_stem": "Brain Stem",
    "left_amygdala": "Left Amygdala",
    "left_caudate": "Left Caudate",
    "left_thalamus": "Left Thalamus",
    "right_amygdala": "Right Amygdala",
    "right_caudate": "Right Caudate",
    "right_thalamus": "Right Thalamus",
    "ctx_lh_rostralanteriorcingulate": "Left rACC",
    "left_putamen": "Left Putamen",
    "right_putamen": "Right Putamen",
    "ctx_rh_rostralanteriorcingulate": "Right rACC",
    "ctx_rh_rostralmiddlefrontal": "Right rMFC",
    "ctx_lh_rostralmiddlefrontal": "Left rMFC",
}

In [ ]:
# Fitting single SVM with all ROIs as predictors
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import svm
from sklearn.metrics import RocCurveDisplay, auc
from sklearn.model_selection import StratifiedKFold, cross_validate

svm_results_dir = postproc_res_dir_ver / "svm"
svm_results_dir.mkdir(exist_ok=True)

n_splits = 10
cv = StratifiedKFold(n_splits=n_splits)
classifier = svm.SVC(kernel="linear", probability=True, random_state=42)
cv_results = cross_validate(
    classifier, X, Y, cv=cv, return_estimator=True, return_indices=True
)


prop_cycle = plt.rcParams["axes.prop_cycle"]
colors = prop_cycle.by_key()["color"]
curve_kwargs_list = [
    dict(alpha=0.3, lw=1, color=colors[fold % len(colors)]) for fold in range(n_splits)
]
names = [f"ROC fold {idx}" for idx in range(n_splits)]

mean_fpr = np.linspace(0, 1, 100)
interp_tprs = []

_, ax = plt.subplots(figsize=(6, 6))
viz = RocCurveDisplay.from_cv_results(
    cv_results,
    X,
    Y,
    ax=ax,
    name="_nolegend_",
    curve_kwargs=curve_kwargs_list,
    plot_chance_level=True,
)

for idx in range(n_splits):
    interp_tpr = np.interp(mean_fpr, viz.fpr[idx], viz.tpr[idx])
    interp_tpr[0] = 0.0
    interp_tprs.append(interp_tpr)

mean_tpr = np.mean(interp_tprs, axis=0)
mean_tpr[-1] = 1.0
mean_auc = auc(mean_fpr, mean_tpr)
std_auc = np.std(viz.roc_auc)

ax.plot(
    mean_fpr,
    mean_tpr,
    color="b",
    label=r"Mean ROC (AUC = %0.2f $\pm$ %0.2f)" % (mean_auc, std_auc),
    lw=2,
    alpha=0.8,
)

std_tpr = np.std(interp_tprs, axis=0)
tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
ax.fill_between(
    mean_fpr,
    tprs_lower,
    tprs_upper,
    color="grey",
    alpha=0.2,
    label=r"$\pm$ 1 std. dev.",
)

target_names = ["HC", "MDD"]

ax.set(
    xlabel="False Positive Rate",
    ylabel="True Positive Rate",
    title=f"Full",
)
print(round(mean_auc, 3), round(std_auc, 3))
ax.legend(loc="lower right", fontsize= 6)
plt.savefig(svm_results_dir / "roc_full.png", dpi = 300, bbox_inches = "tight")
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score
import numpy as np
classifier = svm.SVC(kernel="linear", probability=True, random_state=42)

auc_true = cross_val_score(classifier, X, Y, cv=cv, scoring="roc_auc")
auc_true_mean = auc_true.mean()
print(auc_true_mean)

rng = np.random.default_rng(42)

perm_aucs_list = np.zeros(100)

for i in range(len(perm_aucs_list)):
    y_perm = rng.permutation(Y)
    auc_perm = cross_val_score(
        classifier,
        X,
        y_perm,
        cv=cv,
        scoring="roc_auc"
    ) 
    perm_aucs_list[i] = auc_perm.mean()

C = np.sum(perm_aucs_list >= auc_true_mean)
p = (C + 1) / (len(perm_aucs_list) + 1)

print(p)

In [ ]:
# Fitting SVM for each ROI seperately
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import svm
from sklearn.metrics import RocCurveDisplay, auc
from sklearn.model_selection import StratifiedKFold, cross_validate

# dir = Path("roi_roc_plots")
# dir.mkdir(parents=True, exist_ok=True)

svm_results_dir = postproc_res_dir_ver / "svm"
svm_results_dir.mkdir(exist_ok=True)

n_splits = 10
target_groups = ["HC_test", "MDD"]
target_names = ["HC", "MDD"]
roi_cols = [col for col in data_z.columns if col not in ["subject_id", "group"]]

prop_cycle = plt.rcParams["axes.prop_cycle"]
colors = prop_cycle.by_key()["color"]
curve_kwargs_list = [
    dict(alpha=0.3, lw=1, color=colors[fold % len(colors)]) for fold in range(n_splits)
]
names = [f"ROC fold {idx}" for idx in range(n_splits)]

auc_rows = []

mean_aucs = {}
std_aucs = {}
n_obs = {}

for roi in rois:
    roi_df = data_z.loc[data_z["group"].isin(target_groups), ["group", roi]].dropna().copy()
    class_counts = roi_df["group"].value_counts()

    if len(class_counts) < 2 or class_counts.min() < n_splits:
        continue

    X_roi = roi_df[[roi]].to_numpy(dtype=float)
    n_obs[roi] = len(roi_df)
    Y_roi = roi_df["group"].to_numpy(dtype=str)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    classifier = svm.SVC(kernel="linear", probability=True, random_state=42)
    cv_results = cross_validate(
        classifier, X_roi, Y_roi, cv=cv, return_estimator=True, return_indices=True
    )

    mean_fpr = np.linspace(0, 1, 100)
    interp_tprs = []

    fig, ax = plt.subplots(figsize=(6, 6))

    viz = RocCurveDisplay.from_cv_results(
        cv_results,
        X_roi,
        Y_roi,
        ax=ax,
        name="_nolegend_",
        curve_kwargs=curve_kwargs_list,
        plot_chance_level=True,
    )

    for idx in range(n_splits):
        interp_tpr = np.interp(mean_fpr, viz.fpr[idx], viz.tpr[idx])
        interp_tpr[0] = 0.0
        interp_tprs.append(interp_tpr)

    mean_tpr = np.mean(interp_tprs, axis=0)
    mean_tpr[-1] = 1.0
    mean_auc = auc(mean_fpr, mean_tpr)
    std_auc = np.std(viz.roc_auc)

    mean_aucs[roi] = mean_auc
    std_aucs[roi] = std_auc

    ax.plot(
        mean_fpr,
        mean_tpr,
        color="b",
        label=r"Mean ROC (AUC = %0.2f $\pm$ %0.2f)" % (mean_auc, std_auc),
        lw=2,
        alpha=0.8,
    )

    std_tpr = np.std(interp_tprs, axis=0)
    tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
    tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
    ax.fill_between(
        mean_fpr,
        tprs_lower,
        tprs_upper,
        color="grey",
        alpha=0.2,
        label=r"$\pm$ 1 std. dev.",
    )

    ax.set(
        xlabel="False Positive Rate",
        ylabel="True Positive Rate",
        title=f"{roi_mapping[roi]}",
    )
    ax.legend(loc="lower right", fontsize=6)

    auc_rows.append(
        {
            "roi": roi,
            "auc": mean_auc,
            "std_auc": std_auc,
            "n_subjects": len(Y_roi),
        }
    )

    print(names)
    fig.tight_layout()
    fig.savefig(svm_results_dir / f"roc_{roi_mapping[roi]}.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

# auc_df = pd.DataFrame(auc_rows).sort_values("auc", ascending=False).reset_index(drop=True)
# auc_df


In [ ]:
from sklearn.model_selection import cross_val_score
import numpy as np


for roi in rois:
    roi_df = data_z.loc[data_z["group"].isin(["HC_test", "MDD"]), ["group", roi]].dropna().copy()

    X_roi = roi_df[[roi]].to_numpy(dtype=float)
    Y_roi = roi_df["group"].to_numpy(dtype=str)

    classifier = svm.SVC(kernel="linear", probability=True, random_state=42)
    rng = np.random.default_rng(42)
 
    perm_aucs_list = np.zeros(100)

    auc_true = cross_val_score(classifier, X_roi, Y_roi, cv=cv, scoring="roc_auc")
    auc_true_mean = auc_true.mean()
    # print(auc_true_mean)


    for i in range(len(perm_aucs_list)):
        y_perm = rng.permutation(Y_roi)
        auc_perm = cross_val_score(
            classifier,
            X_roi,
            y_perm,
            cv=cv,
            scoring="roc_auc"
        ) 
        perm_aucs_list[i] = auc_perm.mean()

    C = np.sum(perm_aucs_list >= auc_true_mean)
    p = (C + 1) / (len(perm_aucs_list) + 1)

    print(roi)
    if p <0.05:
        print("p:", p, "<=============")
    else:
        print("p:", p)

In [ ]:

for roi in rois:
    print("ROI:", roi_mapping[roi])
    print("mean auc:", round(mean_aucs[roi], 3))
    print("std auc:", round(std_aucs[roi], 3))
    print("n observation:", n_obs[roi])
    print()


## Extreme deviation counts

In [ ]:

extreme_deviations = data_z.copy()
extreme_deviations = extreme_deviations[extreme_deviations["group"] != "HC_train"]
extreme_mask = (extreme_deviations[rois] > 2) | (extreme_deviations[rois] < -2)

extreme_deviations = (
    extreme_mask
    .groupby(extreme_deviations["group"])
    .sum()
)


extreme_positive = data_z.copy()
extreme_positive = extreme_positive[extreme_positive["group"] != "HC_train"]
extreme_pos_mask = (extreme_positive[rois] > 2)


extreme_positive = (
    extreme_pos_mask
    .groupby(extreme_positive["group"])
    .sum()
)


extreme_negative = data_z.copy()
extreme_negative = extreme_negative[extreme_negative["group"] != "HC_train"]
extreme_pos_mask = (extreme_negative[rois] < -2)


extreme_negative = (
    extreme_pos_mask
    .groupby(extreme_negative["group"])
    .sum()
)




In [ ]:
roi_order = [
    "brain_stem",
    "left_amygdala",
    "right_amygdala",
    "left_caudate",
    "right_caudate",
    "left_thalamus",
    "right_thalamus",
    "left_putamen",
    "right_putamen",
    "ctx_lh_rostralanteriorcingulate",
    "ctx_rh_rostralanteriorcingulate",
    "ctx_lh_rostralmiddlefrontal",
    "ctx_rh_rostralmiddlefrontal",
]

roi_labels = [
    "Brain stem",
    "Left amygdala",
    "Right amygdala",
    "Left caudate",
    "Right caudate",
    "Left thalamus",
    "Right thalamus",
    "Left putamen",
    "Right putamen",
    "Left rACC",
    "Right rACC",
    "Left rMFC",
    "Right rMFC",
]

deviation_counts_res_dir = postproc_res_dir_ver / "dev_counts"
deviation_counts_res_dir.mkdir(exist_ok=True)

hc = extreme_deviations.loc["HC_test"].reindex(roi_order)
hc_pos = extreme_positive.loc["HC_test"].reindex(roi_order)
hc_neg = extreme_negative.loc["HC_test"].reindex(roi_order)
mdd = extreme_deviations.loc["MDD"].reindex(roi_order)
mdd_pos = extreme_positive.loc["MDD"].reindex(roi_order)
mdd_neg = extreme_negative.loc["MDD"].reindex(roi_order)

x = np.arange(len(roi_order))

fig, ax = plt.subplots(figsize=(14, 6))

# ax.bar(x, hc.values, color="steelblue", label="Control")
# ax.bar(x, -mdd.values, color="indianred", label="MDD")
# ax.bar(x - 0.2, hc_pos.values, color = "blue", label="control positive", width=0.4)
# ax.bar(x + 0.2, hc_neg.values, color = "purple", label="control negative", width=0.4)
# ax.bar(x - 0.2, -mdd_pos.values, color = "pink", label="MDD positive", width=0.4)
# ax.bar(x + 0.2, -mdd_neg.values, color = "yellow", label="MDD negative", width=0.4)

ax.bar(x, hc.values, color="lightsteelblue", label="Control total")
ax.bar(x, -mdd.values, color="lightcoral", label="MDD total")

ax.bar(x - 0.2, hc_pos.values, color="royalblue", 
       label="control positive", width=0.4, edgecolor = "black", linewidth = 0.7)
ax.bar(x + 0.2, hc_neg.values, color="skyblue", 
       label="control negative", width=0.4, edgecolor = "black", linewidth = 0.7)

ax.bar(x - 0.2, -mdd_pos.values, color="firebrick", 
       label="MDD positive", width=0.4, edgecolor = "black", linewidth = 0.7)
ax.bar(x + 0.2, -mdd_neg.values, color="lightpink", 
       label="MDD negative", width=0.4, edgecolor = "black", linewidth = 0.7)

ax.axhline(0, color="black", linewidth=0.8)

y_ticks = [i for i in range(-6,8, 2)]

ax.set_xticks(x)
ax.set_yticks(y_ticks)
ax.set_yticklabels([abs(x) for x in y_ticks])
ax.set_xticklabels(roi_labels, rotation=60)
ax.set_ylabel("Extreme deviations")
ax.set_xlabel("ROI")
ax.set_title("Extreme deviations by ROI")
ax.legend()

plt.tight_layout()
# plt.savefig(deviation_counts_res_dir / "roi_coded.png", dpi = 300, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np

extreme_dev_dict = {"MDD": {}, "HC_test": {}}

for group in ["MDD", "HC_test"]:
    extreme_deviations = data_z.copy()
    extreme_deviations = extreme_deviations[extreme_deviations["group"] == group]
    extreme_mask = (extreme_deviations[rois] > 2) | (extreme_deviations[rois] < -2)


    extreme_deviations = (
        extreme_mask
        .groupby(extreme_deviations["subject_id"])[rois]
        .sum()
    )

    extreme_deviations["total"] = extreme_deviations[rois].apply(np.sum, axis=1)
    extreme_dev_dict[group]["extr_dev"] = extreme_deviations


    extreme_positive = data_z.copy()
    extreme_positive = extreme_positive[extreme_positive["group"] == group]
    extreme_pos_mask = (extreme_positive[rois] > 2)


    extreme_positive = (
        extreme_pos_mask
        .groupby(extreme_positive["subject_id"])
        .sum()
    )
    extreme_positive["total"] = extreme_positive[rois].apply(np.sum, axis=1)
    extreme_dev_dict[group]["extr_pos"] = extreme_positive

    extreme_negative = data_z.copy()
    extreme_negative = extreme_negative[extreme_negative["group"] == group]
    extreme_negative_mask = (extreme_negative[rois] < -2)


    extreme_negative = (
        extreme_negative_mask
        .groupby(extreme_negative["subject_id"])
        .sum()
    )
    extreme_negative["total"] = extreme_negative[rois].apply(np.sum, axis=1)
    extreme_dev_dict[group]["extr_neg"] = extreme_negative


In [ ]:
mdd_extr = extreme_dev_dict["MDD"]["extr_dev"]
hc_extr = extreme_dev_dict["HC_test"]["extr_dev"]
hc_pos = extreme_dev_dict["HC_test"]["extr_pos"]
mdd_pos = extreme_dev_dict["MDD"]["extr_pos"]
hc_neg = extreme_dev_dict["HC_test"]["extr_neg"]
mdd_neg = extreme_dev_dict["MDD"]["extr_neg"]

In [ ]:
from scipy.stats import mannwhitneyu, ttest_ind

print(mannwhitneyu(hc_extr["total"], mdd_extr["total"], alternative="two-sided"))
print(mannwhitneyu(hc_neg["total"], mdd_neg["total"], alternative="two-sided"))
print(mannwhitneyu(hc_pos["total"], mdd_pos["total"], alternative="two-sided"))

In [ ]:
roi_mapping = {
    "brain_stem": "Brain Stem",
    "left_amygdala": "Left Amygdala",
    "left_caudate": "Left Caudate",
    "left_thalamus": "Left Thalamus",
    "right_amygdala": "Right Amygdala",
    "right_caudate": "Right Caudate",
    "right_thalamus": "Right Thalamus",
    "ctx_lh_rostralanteriorcingulate": "Left rACC",
    "left_putamen": "Left Putamen",
    "right_putamen": "Right Putamen",
    "ctx_rh_rostralanteriorcingulate": "Right rACC",
    "ctx_rh_rostralmiddlefrontal": "Right rMFC",
    "ctx_lh_rostralmiddlefrontal": "Left rMFC",
}

In [ ]:
from scipy.stats import ttest_ind

for roi in rois:
    test = data_z[data_z["group"] == "HC_test"][roi].dropna()
    mdd = data_z[data_z["group"] == "MDD"][roi].dropna()
    whitney_res = mannwhitneyu(test, mdd, alternative = "two-sided")
    ttest_res = ttest_ind(test, mdd, equal_var=False)
    print(roi_mapping[roi])
    # print("whitney: ", whitney_res.pvalue)
    print(f"t-test: t={ttest_res.statistic}, p={ttest_res.pvalue}, df={ttest_res.df}")
    print("HC mean:", test.mean())
    print("HC std:", test.std())
    print("MDD mean:", mdd.mean())
    print("MDD std:", mdd.std())
    print()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

deviation_counts_res_dir = postproc_res_dir_ver / "dev_counts"
deviation_counts_res_dir.mkdir(exist_ok=True)


mdd_extr = extreme_dev_dict["MDD"]["extr_dev"]
mdd_extr["group"] = "MDD"
hc_extr = extreme_dev_dict["HC_test"]["extr_dev"]
hc_extr["group"] = "HC_test"
hc_pos = extreme_dev_dict["HC_test"]["extr_pos"]
hc_pos["group"] = "HC_test"
mdd_pos = extreme_dev_dict["MDD"]["extr_pos"]
mdd_pos["group"] = "MDD"
hc_neg = extreme_dev_dict["HC_test"]["extr_neg"]
hc_neg["group"] = "HC_test"
mdd_neg = extreme_dev_dict["MDD"]["extr_neg"]
mdd_neg["group"] = "MDD"

sns.set_theme(style="darkgrid")


pos_df = pd.concat([hc_pos, mdd_pos], ignore_index=True)
neg_df = pd.concat([hc_neg, mdd_neg], ignore_index=True)


pos_test = mannwhitneyu(pos_df[pos_df["group"] == "HC_test"]["total"], pos_df[pos_df["group"] == "MDD"]["total"], alternative="two-sided")
neg_test = mannwhitneyu(neg_df[neg_df["group"] == "HC_test"]["total"], neg_df[neg_df["group"] == "MDD"]["total"], alternative="two-sided")

fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)

sns.violinplot(data=pos_df, x="group", y="total", hue = "group", palette=["Blue", "Red"], ax=axes[0])

axes[0].set_ylim(0, 10)
axes[0].set_title( f"Extreme positives (z>2)\nstat = {pos_test.statistic:.2f}, p = {pos_test.pvalue:.3f}")
axes[0].set_ylabel("Count")
axes[0].set_xlabel("")
axes[0].set_xticklabels(["HC", "MDD"])

sns.violinplot(data=neg_df, x="group", y="total", hue = "group", palette=["Blue", "Red"], ax=axes[1])

axes[1].set_ylim(0, 10)
axes[1].set_title(
    f"Extreme negatives (z<-2)\nstat = {neg_test.statistic:.2f}, p = {neg_test.pvalue:.3f}"
)
axes[1].set_ylabel("")
axes[1].set_xlabel("")
axes[1].set_xticklabels(["HC", "MDD"])

plt.tight_layout()
# plt.savefig(deviation_counts_res_dir / "total_extr_dev.png", dpi = 300, bbox_inches="tight")
plt.show()

## Depression scores and deviation scores

In [ ]:
# Read original data with subject information
original_data = pd.read_excel("data/data_all_sites_aggregated2.xlsx")

# Filter data to keep relevant rows
original_data = original_data[original_data["session"] == "ses-baseline"]
original_data = original_data[original_data["diagnosis"].isin(["HC", "MDD"])]


In [ ]:
# Merge subject information into z score dataframe

columns_to_merge = ["id", "participant_id", "site", # "id" and "particpant_id" are required
                    "age", "diagnosis", "sex", 
                    "madrs", "ham_d_24", "ham_d_17",
                      "major_depression_inventory"]

data_z2 = data_z.merge(
    original_data[columns_to_merge],
    left_on = "subject_id",
    right_on = "id",
    how = "left",
    validate = "many_to_one"
)
data_z2.shape

In [ ]:
data_z2.sample(5, random_state=42)

print("All: ")
print(data_z2["madrs"].notna().sum())
print(data_z2["ham_d_17"].notna().sum())
print(data_z2["ham_d_24"].notna().sum())

print("Only MDD group:")
print(data_z2[data_z2["diagnosis"] == "MDD"]["madrs"].notna().sum())
print(data_z2[data_z2["diagnosis"] == "MDD"]["ham_d_17"].notna().sum())
print(data_z2[data_z2["diagnosis"] == "MDD"]["ham_d_24"].notna().sum())



### Depression score conversions

Methods from Heo, Murphy & Meyers (2007)

In [ ]:
# Converting ham_d_24 to ham_d_17
mask = data_z2["ham_d_17"].isna() & data_z2["ham_d_24"].notna()
data_z2.loc[mask, "ham_d_17"] = (17 * (data_z2.loc[mask, "ham_d_24"]/24)).round()

# Converting madrs to ham_d_17
mask = data_z2["ham_d_17"].isna() & data_z2["madrs"].notna()
data_z2.loc[mask, "ham_d_17"] = (-2.68 + 0.86 * data_z2.loc[mask, "madrs"]).round().clip(lower=0)

In [ ]:
data_z2.sample(5, random_state=42)

print("All: ")
print(data_z2["madrs"].notna().sum())
print(data_z2["ham_d_17"].notna().sum())
print(data_z2["ham_d_24"].notna().sum())

print("Only MDD group:")
print(data_z2[data_z2["diagnosis"] == "MDD"]["madrs"].notna().sum())
print(data_z2[data_z2["diagnosis"] == "MDD"]["ham_d_17"].notna().sum())
print(data_z2[data_z2["diagnosis"] == "MDD"]["ham_d_24"].notna().sum())



In [ ]:
data_z_hamd17 = data_z2.dropna(subset=["ham_d_17"])

data_z_hamd17 = data_z_hamd17[data_z_hamd17["group"] == "MDD"]
print(data_z_hamd17.shape)

In [ ]:

rois = ['brain_stem', 'left_amygdala', 'left_caudate',
       'left_thalamus', 'right_amygdala', 'right_caudate', 'right_thalamus',
       'ctx_lh_rostralanteriorcingulate', 'left_putamen', 'right_putamen',
       'ctx_rh_rostralanteriorcingulate', 'ctx_rh_rostralmiddlefrontal',
       'ctx_lh_rostralmiddlefrontal']


In [ ]:
print(data_z_hamd17.columns)

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

formula = "ham_d_17 ~ " + " + ".join(rois) + " + C(sex)"
print(formula)

lin_model = smf.ols(formula=formula, data=data_z_hamd17).fit()
print(lin_model.summary())


In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
# f(deviation) = depression score + sex_effect

hamd_linreg_dir = postproc_res_dir_ver / "linreg"
hamd_linreg_dir.mkdir(exist_ok=True)

# frontals = ['ctx_rh_rostralmiddlefrontal', 'ctx_lh_rostralmiddlefrontal']

models_hamd17 = {}
for roi in rois:
    print(roi)
    intermediary_df = data_z_hamd17.dropna(subset=[roi])
    # intermediary_df = intermediary_df[intermediary_df[roi] < 5]
    x_hamd17 = intermediary_df[roi].to_numpy().reshape(-1,1)
    X_hamd17 = sm.add_constant(x_hamd17)
    Y_hamd17 = intermediary_df["ham_d_17"].to_numpy()

    model = smf.ols(formula = "ham_d_17 ~ " + str(roi) + "+ C(sex)", data = data_z_hamd17).fit()
    models_hamd17[roi] = model

    # HAM-D-17 plot
    intermediary_df = data_z_hamd17.dropna(subset=[roi])
    intermediary_df = intermediary_df[intermediary_df[roi] < 5]
    x_hamd17 = intermediary_df[roi].to_numpy().reshape(-1,1)
    Y_hamd17 = intermediary_df["ham_d_17"].to_numpy()


    x_line = np.linspace(x_hamd17.min(), x_hamd17.max(), 100)
    X_line_f = pd.DataFrame({roi: x_line,
                           "sex": data_z_hamd17["sex"].unique()[0]})
    X_line_m = pd.DataFrame({roi: x_line,
                           "sex": data_z_hamd17["sex"].unique()[1]})

    plt.figure(figsize=(6, 4))
    plt.scatter(x_hamd17, Y_hamd17, alpha=0.7)
    plt.plot(x_line, model.predict(X_line_f), color="red")
    plt.plot(x_line, model.predict(X_line_m), color="blue")
    plt.xlabel(f"Deviation score")
    plt.ylabel("HAM-D-17 score")
    plt.title(f"{roi_mapping[roi]}")
    plt.savefig(hamd_linreg_dir / f"{roi_mapping[roi]}_2", dpi = 300, bbox_inches="tight")
    plt.show()


In [ ]:

for roi, model_hamd17 in models_hamd17.items():
    r2_string = f"HAM-D-17 R2: {model_hamd17.rsquared}"
    if model_hamd17.rsquared > 0.1:
        r2_string = r2_string + "<======="
    p_string1 = f"HAM-D-17 p: {model_hamd17.pvalues[roi]}"
    if model_hamd17.pvalues[roi] <= 0.05:
        p_string1 = p_string1 + "<======="
    p_string2 = f"Ham-D-17 P: {model_hamd17.pvalues["C(sex)[T.M]"]}"
    if model_hamd17.pvalues["C(sex)[T.M]"] <= 0.05:
        p_string2 = p_string2 + "<======="
    print("\n", roi)
    print(r2_string)
    print("roi:\n", p_string1)
    print("sex:\n", p_string2)

In [ ]:
for key, val in models_hamd17.items():
    print(roi_mapping[key])
    print("Gender effect:", round(val.params["C(sex)[T.M]"], 2))
    print("slope:", round(val.params[key], 2))
    print("rsquard:", round(val.rsquared, 4))
    print("observations:", int(val.nobs))
    print("-----------")